# Module 6: Statistics for Machine Learning

**Topics Covered:**
- Descriptive statistics (mean, variance, skewness, kurtosis)
- Probability distributions (normal, binomial, Poisson)
- Central Limit Theorem
- Hypothesis testing (t-test, chi-squared)
- Correlation and covariance
- Bayesian intuition
- Bias-Variance tradeoff (quantified)

> Statistics is the language ML speaks. Understanding it means you can interpret model outputs, not just compute them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
print("Libraries loaded.")

---
## 1. Descriptive Statistics

In [ ]:
# Simulate feature data
np.random.seed(42)
income = np.random.lognormal(mean=10.8, sigma=0.6, size=1000)  # right-skewed
age    = np.random.normal(loc=38, scale=12, size=1000).clip(18, 75)

def describe_feature(name, data):
    print(f"\n=== {name} ===")
    print(f"  Mean:      {np.mean(data):>12.2f}")
    print(f"  Median:    {np.median(data):>12.2f}")
    print(f"  Std:       {np.std(data):>12.2f}")
    print(f"  Variance:  {np.var(data):>12.2f}")
    print(f"  Skewness:  {stats.skew(data):>12.4f}")
    print(f"  Kurtosis:  {stats.kurtosis(data):>12.4f}")
    print(f"  IQR:       {np.percentile(data,75)-np.percentile(data,25):>12.2f}")

describe_feature('Income (log-normal)', income)
describe_feature('Age (normal)', age)

In [ ]:
# Visualize skewness
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, name in zip(axes, [income, age], ['Income (right-skewed)', 'Age (normal)']):
    ax.hist(data, bins=40, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(np.mean(data), color='red', linestyle='--', label=f'Mean={np.mean(data):.0f}')
    ax.axvline(np.median(data), color='orange', linestyle='--', label=f'Median={np.median(data):.0f}')
    ax.set_title(name); ax.legend()

plt.tight_layout()
plt.show()

### Exercise 6.1 — Descriptive Statistics
Load the following feature and:
1. Compute mean, median, std, skewness
2. Identify which is more appropriate — mean or median — and explain why
3. Compute the 5th and 95th percentiles for outlier detection
4. Count how many values fall outside the 5th-95th percentile range

In [ ]:
np.random.seed(1)
housing_prices = np.concatenate([
    np.random.normal(250000, 50000, 900),  # typical homes
    np.random.normal(2000000, 500000, 100) # luxury homes
])
# YOUR CODE HERE


In [ ]:
# SOLUTION
np.random.seed(1)
housing_prices = np.concatenate([
    np.random.normal(250000, 50000, 900),
    np.random.normal(2000000, 500000, 100)
])

mean   = np.mean(housing_prices)
median = np.median(housing_prices)
std    = np.std(housing_prices)
skew   = stats.skew(housing_prices)

p5, p95 = np.percentile(housing_prices, [5, 95])
outlier_count = np.sum((housing_prices < p5) | (housing_prices > p95))

print(f"Mean:   ${mean:>12,.0f}")
print(f"Median: ${median:>12,.0f}")
print(f"Std:    ${std:>12,.0f}")
print(f"Skewness: {skew:.3f} → strongly right-skewed → MEDIAN is better representative")
print(f"\n5th percentile:  ${p5:>12,.0f}")
print(f"95th percentile: ${p95:>12,.0f}")
print(f"Outliers: {outlier_count} ({outlier_count/len(housing_prices):.1%})")

---
## 2. Probability Distributions

In [ ]:
# Normal distribution — fundamental to ML assumptions
from scipy.stats import norm, binom, poisson

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Normal
x = np.linspace(-4, 4, 200)
for mu, sigma, ls in [(0, 1, '-'), (0, 2, '--'), (1, 1, ':')]:
    axes[0].plot(x, norm.pdf(x, mu, sigma), label=f'μ={mu}, σ={sigma}', lw=2, ls=ls)
axes[0].set_title('Normal Distributions'); axes[0].legend(fontsize=8)

# 2. Binomial (success in n trials)
k = np.arange(0, 21)
for n_trials, p in [(20, 0.3), (20, 0.5), (20, 0.7)]:
    axes[1].plot(k, binom.pmf(k, n_trials, p), 'o-', label=f'p={p}', ms=5)
axes[1].set_title('Binomial PMF (n=20)'); axes[1].legend(fontsize=8)
axes[1].set_xlabel('# successes')

# 3. Poisson (rare events)
k = np.arange(0, 20)
for lam in [1, 3, 6]:
    axes[2].plot(k, poisson.pmf(k, lam), 'o-', label=f'λ={lam}', ms=5)
axes[2].set_title('Poisson PMF'); axes[2].legend(fontsize=8)
axes[2].set_xlabel('Events')

plt.tight_layout()
plt.show()

In [ ]:
# Z-score normalization and the empirical rule
data = np.random.normal(100, 15, 10000)  # IQ-like scores

z_scores = (data - data.mean()) / data.std()

within_1std = np.mean(np.abs(z_scores) <= 1)
within_2std = np.mean(np.abs(z_scores) <= 2)
within_3std = np.mean(np.abs(z_scores) <= 3)

print("Empirical Rule (should be ~68/95/99.7%):")
print(f"Within 1σ: {within_1std:.2%}")
print(f"Within 2σ: {within_2std:.2%}")
print(f"Within 3σ: {within_3std:.2%}")

---
## 3. Central Limit Theorem

In [ ]:
# CLT: sample means approach normal dist regardless of population distribution
# This underpins confidence intervals, hypothesis tests, and many ML assumptions

population = np.random.exponential(scale=5, size=100000)  # skewed population

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, sample_size in zip(axes, [1, 5, 30, 100]):
    sample_means = [np.mean(np.random.choice(population, sample_size)) for _ in range(2000)]
    ax.hist(sample_means, bins=40, density=True, color='steelblue', edgecolor='white', alpha=0.7)
    ax.set_title(f'n={sample_size}\nMean={np.mean(sample_means):.2f}')

axes[0].set_xlabel('Sample mean (n=1 = population)')
plt.suptitle('Central Limit Theorem: Distribution of Sample Means', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Hypothesis Testing

In [ ]:
# A/B Testing — model A vs model B accuracy
np.random.seed(42)

# Simulate 30 cross-val fold accuracies for two models
model_a = np.random.normal(0.82, 0.03, 30)
model_b = np.random.normal(0.85, 0.03, 30)

# Paired t-test (same data, different models)
t_stat, p_value = stats.ttest_rel(model_a, model_b)

print(f"Model A: mean={model_a.mean():.4f}, std={model_a.std():.4f}")
print(f"Model B: mean={model_b.mean():.4f}, std={model_b.std():.4f}")
print(f"\nPaired t-test: t={t_stat:.4f}, p={p_value:.4f}")
print(f"Significant at α=0.05? {'YES' if p_value < 0.05 else 'NO'}")

# Effect size (Cohen's d)
pooled_std = np.sqrt((model_a.std()**2 + model_b.std()**2) / 2)
cohen_d = (model_b.mean() - model_a.mean()) / pooled_std
print(f"Cohen's d: {cohen_d:.4f} ({'small' if abs(cohen_d)<0.5 else 'medium' if abs(cohen_d)<0.8 else 'large'} effect)")

In [ ]:
# Chi-squared test — feature independence from target
from scipy.stats import chi2_contingency

# Contingency table: Education vs Default
contingency = np.array([
    [50, 20],   # High school: [no default, default]
    [80, 15],   # Bachelor
    [40,  5],   # Master
    [20,  2],   # PhD
])

chi2, p_val, dof, expected = chi2_contingency(contingency)

print("Chi-squared Test: Education vs Default")
print(f"  χ² = {chi2:.4f}")
print(f"  p-value = {p_val:.4f}")
print(f"  Degrees of freedom = {dof}")
print(f"  Are education and default independent? {'NO (dependent)' if p_val < 0.05 else 'YES (independent)'}")

### Exercise 6.2 — Hypothesis Testing
You trained the same model 5 times with different random seeds on the same data.

```python
scores_default = [0.84, 0.83, 0.85, 0.82, 0.84]
scores_tuned   = [0.87, 0.89, 0.88, 0.86, 0.90]
```

1. Perform a **paired t-test** to see if tuning significantly improved results
2. Compute a **95% confidence interval** for the mean improvement
3. Visualize both distributions with box plots side-by-side
4. Interpret: should you report tuning as beneficial?

In [ ]:
scores_default = np.array([0.84, 0.83, 0.85, 0.82, 0.84])
scores_tuned   = np.array([0.87, 0.89, 0.88, 0.86, 0.90])
# YOUR CODE HERE


In [ ]:
# SOLUTION
scores_default = np.array([0.84, 0.83, 0.85, 0.82, 0.84])
scores_tuned   = np.array([0.87, 0.89, 0.88, 0.86, 0.90])

# 1. Paired t-test
t_stat, p_val = stats.ttest_rel(scores_default, scores_tuned)
print(f"Paired t-test: t={t_stat:.4f}, p={p_val:.4f}")

# 2. CI for mean improvement
diff = scores_tuned - scores_default
mean_diff = diff.mean()
se = stats.sem(diff)
ci = stats.t.interval(0.95, df=len(diff)-1, loc=mean_diff, scale=se)
print(f"Mean improvement: {mean_diff:.4f}")
print(f"95% CI: ({ci[0]:.4f}, {ci[1]:.4f})")

# 3. Visualization
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([scores_default, scores_tuned], labels=['Default', 'Tuned'])
ax.set_ylabel('Accuracy')
ax.set_title(f'Model Comparison (p={p_val:.4f})')
plt.tight_layout()
plt.show()

# 4. Interpretation
print(f"\nInterpretation: p={p_val:.4f} {'< 0.05' if p_val < 0.05 else '≥ 0.05'}")
if p_val < 0.05:
    print("The improvement is statistically significant — YES, tuning helps.")
else:
    print("Not significant with n=5. Consider more runs before reporting.")

---
## 5. Correlation & Covariance

In [ ]:
# Pearson vs Spearman correlation
np.random.seed(42)
n = 200
x = np.random.randn(n)
y_linear    = 2 * x + np.random.randn(n) * 0.5
y_nonlinear = x**2 + np.random.randn(n) * 0.5  # nonlinear relationship

for name, y in [('Linear', y_linear), ('Nonlinear (quadratic)', y_nonlinear)]:
    pearson_r, pearson_p   = stats.pearsonr(x, y)
    spearman_r, spearman_p = stats.spearmanr(x, y)
    print(f"{name}:")
    print(f"  Pearson r={pearson_r:.3f} (p={pearson_p:.4f})")
    print(f"  Spearman r={spearman_r:.3f} (p={spearman_p:.4f})")

# Key insight: Spearman is better for monotonic non-linear relationships

---
## 6. Bias-Variance Tradeoff (Quantified)

In [ ]:
# Simulate bias-variance tradeoff
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

def true_function(x):
    return np.sin(x * np.pi)

np.random.seed(42)
x_train_pts = np.random.uniform(-1, 1, 10)
y_train_pts = true_function(x_train_pts) + np.random.randn(10) * 0.3
x_test = np.linspace(-1, 1, 200)

degrees = [1, 3, 9]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, deg in zip(axes, degrees):
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=deg)),
        ('reg',  LinearRegression())
    ])
    model.fit(x_train_pts.reshape(-1,1), y_train_pts)
    y_pred = model.predict(x_test.reshape(-1,1))
    
    train_preds = model.predict(x_train_pts.reshape(-1,1))
    train_mse = np.mean((y_train_pts - train_preds)**2)
    test_mse  = np.mean((true_function(x_test) - y_pred)**2)

    ax.scatter(x_train_pts, y_train_pts, color='black', zorder=5, label='Train data')
    ax.plot(x_test, true_function(x_test), '--', color='gray', label='True function', linewidth=2)
    ax.plot(x_test, y_pred, color='steelblue', label='Model', linewidth=2)
    
    status = 'Underfitting' if deg==1 else ('Good' if deg==3 else 'Overfitting')
    ax.set_title(f'Degree {deg} Polynomial\n({status})\nTrain MSE={train_mse:.3f} | Test MSE={test_mse:.3f}')
    ax.legend(fontsize=7)
    ax.set_ylim(-2, 2)

plt.tight_layout()
plt.show()

---
## Challenge: Bootstrap Confidence Intervals

Bootstrap is a powerful non-parametric technique to estimate uncertainty in model metrics.

1. Generate 500 test predictions and ground truth
2. Compute accuracy on the full test set
3. Use **bootstrap resampling** (1000 iterations, with replacement) to estimate the **95% CI for accuracy**
4. Plot the bootstrap distribution with the CI highlighted
5. Compare: if two models have overlapping CIs, can you conclude they're different?

In [ ]:
np.random.seed(42)
n_test = 500
y_true = np.random.choice([0, 1], n_test)
# Model A: 80% accurate
y_pred_a = np.where(np.random.rand(n_test) < 0.80, y_true, 1 - y_true)
# Model B: 83% accurate  
y_pred_b = np.where(np.random.rand(n_test) < 0.83, y_true, 1 - y_true)

# YOUR CODE HERE


In [ ]:
# SOLUTION
np.random.seed(42)
n_test = 500
y_true = np.random.choice([0, 1], n_test)
y_pred_a = np.where(np.random.rand(n_test) < 0.80, y_true, 1 - y_true)
y_pred_b = np.where(np.random.rand(n_test) < 0.83, y_true, 1 - y_true)

def bootstrap_accuracy(y_true, y_pred, n_iterations=1000):
    accs = []
    n = len(y_true)
    for _ in range(n_iterations):
        idx = np.random.randint(0, n, n)
        acc = np.mean(y_true[idx] == y_pred[idx])
        accs.append(acc)
    return np.array(accs)

boot_a = bootstrap_accuracy(y_true, y_pred_a)
boot_b = bootstrap_accuracy(y_true, y_pred_b)

ci_a = np.percentile(boot_a, [2.5, 97.5])
ci_b = np.percentile(boot_b, [2.5, 97.5])

print(f"Model A accuracy: {np.mean(y_true==y_pred_a):.3f} | 95% CI: [{ci_a[0]:.3f}, {ci_a[1]:.3f}]")
print(f"Model B accuracy: {np.mean(y_true==y_pred_b):.3f} | 95% CI: [{ci_b[0]:.3f}, {ci_b[1]:.3f}]")

overlap = ci_a[1] > ci_b[0]
print(f"CIs overlap: {overlap} → {'Cannot conclude B is significantly better' if overlap else 'B is significantly better'}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(boot_a, bins=40, alpha=0.6, color='steelblue', label='Model A')
ax.hist(boot_b, bins=40, alpha=0.6, color='coral', label='Model B')
for ci, color in [(ci_a, 'steelblue'), (ci_b, 'coral')]:
    ax.axvline(ci[0], color=color, linestyle='--', alpha=0.8)
    ax.axvline(ci[1], color=color, linestyle='--', alpha=0.8)
ax.set_xlabel('Accuracy'); ax.set_title('Bootstrap Distribution of Accuracy (95% CIs shown)')
ax.legend()
plt.tight_layout()
plt.show()